# Black Bird APK patcher for Google Colab

Patches **NieR Re[in]carnation** (Global `com.square_enix.android_googleplay.nierspww` or Japan `com.square_enix.android_googleplay.nierspjp`) to connect to a private server. The official service shut down in 2024 and the game cannot be played officially in any form.

Based on version **3.7.1** (latest APK release). Use the Global version. The Japanese release is untested and its offsets are not verified, so it may not work.

Installs tools, configures the target server, accepts an APK (upload or URL), patches it for a private server, rebuilds, signs, and downloads the result.

**Patches**
- `global-metadata.dat` — rewrite IL2CPP string literals (URLs and hostnames)
- `libil2cpp.so` — ARM64 binary patches: SSL bypass, encryption passthrough, plain Octo list, route in-app purchases to the private server (there is no Google Play store backend anymore), gRPC port override
- `AndroidManifest.xml` — set `extractNativeLibs` to true (always), add `networkSecurityConfig` for cleartext HTTP
- `res/xml/network_security_config.xml` — allow cleartext traffic
- Tracking (optional, on by default): turn off Adjust, Firebase Analytics and Crashlytics, Facebook app events and the unused ad SDKs (`libil2cpp.so` and `AndroidManifest.xml`)

**Quick Instructions**
- Configure `protocol`, `server_host`, and optional ports below.
- Leave `disable_tracking` ticked to turn off the game's tracking and analytics SDKs.
- Set `apk_source` to `upload` to upload your own APK, or `url` to download from a URL (`apk_url`).
- Run the single code cell. It installs tools, patches, and downloads the result.

**Notes**
- Choose `http` to enable cleartext network patches; choose `https` to skip the cleartext changes.
- Replacements must not be longer than the original strings; use short hostnames or omit the port if needed.
- `grpc_port` defaults to 443 when left empty. Only set if your server uses a non-standard gRPC port.

In [ ]:
# ── Configuration ────────────────────────────────────────────────────────────
protocol = 'http'  #@param ['http', 'https']
server_host = '192.168.1.1'  #@param {type:'string'}
http_port = ''  #@param {type:'string'}
grpc_port = ''  #@param {type:'string'}
apk_source = 'upload'  #@param ['upload', 'url']
apk_url = ''  #@param {type:'string'}
disable_tracking = True  #@param {type:'boolean'}

# ── Imports & constants ───────────────────────────────────────────────────────
import os
import re
import shutil
import stat
import struct
import subprocess
import urllib.parse
from dataclasses import dataclass
from pathlib import Path

WORK_ROOT = Path('/content/apk_patch_work')
TOOLS_DIR = WORK_ROOT / 'tools'
ANDROID_SDK_ROOT = TOOLS_DIR / 'android-sdk'
APKTOOL_VERSION = '3.0.1'
ANDROID_BUILD_TOOLS = '36.1.0'
ANDROID_CMDLINE_TOOLS_URL = 'https://dl.google.com/android/repository/commandlinetools-linux-11076708_latest.zip'

GAME_NAME = 'nierreincarnation'

METADATA_MAGIC = 0xFAB11BAF
HDR_STRING_LITERAL_OFF = 8
HDR_STRING_LITERAL_DATA_OFF = 16

MOV_X0_0  = struct.pack('<I', 0xD2800000)   # mov x0, #0
MOV_X0_X1 = struct.pack('<I', 0xAA0103E0)   # mov x0, x1
MOV_W0_1  = struct.pack('<I', 0x52800020)   # mov w0, #1
NOP       = struct.pack('<I', 0xD503201F)   # nop
RET       = struct.pack('<I', 0xD65F03C0)   # ret
YEAR_9999 = struct.pack('<I', 0x5284E1E1)   # movz w1, #0x270f (year 9999 literal)

BASE_IL2CPP_PATCHES = [
    {
        'name': 'ToNativeCredentials',
        'desc': 'SSL bypass — return NULL to force insecure gRPC channel',
        'rva': 0x35C8670,
        'bytes': MOV_X0_0 + RET,
    },
    {
        'name': 'HandleNet.Encrypt',
        'desc': 'encryption passthrough — return payload as-is',
        'rva': 0x279410C,
        'bytes': MOV_X0_X1 + RET,
    },
    {
        'name': 'HandleNet.Decrypt',
        'desc': 'decryption passthrough — return receivedMessage as-is',
        'rva': 0x279420C,
        'bytes': MOV_X0_X1 + RET,
    },
    {
        'name': 'OctoManager.Internal.GetListAes',
        'desc': 'Octo list: force plain list (return false = no AES); server serves raw list.bin',
        'rva': 0x4C27038,
        'bytes': MOV_X0_0 + RET,
    },
    {
        'name': 'TitleScreen.ForceYear9999',
        'desc': 'Replace EoS year 2024 with 9999',
        'rva': 0x2F11E7C,
        'bytes': YEAR_9999,
    },
    {
        'name': 'PurchaseRealProductAsync.MoveNext (inlined IsInitialized check)',
        'desc': 'Purchases to private server — NOP the cbz that branches to PurchasingUnavailable when Google Play billing is not initialized',
        'rva': 0x2831CA8,
        'bytes': NOP,
    },
    {
        'name': 'Purchaser.IsExistProduct',
        'desc': 'Purchases to private server — always report product as existing (no Google Play store to query)',
        'rva': 0x282CE78,
        'bytes': MOV_W0_1 + RET,
    },
    {
        'name': 'Purchaser.<BuyProduct>d__24.MoveNext (null _storeController)',
        'desc': 'Purchases to private server — redirect null _storeController from NRE to cancelled-return path',
        'rva': 0x2834028,
        'bytes': struct.pack('<I', 0xB4001675),
    },
    {
        'name': 'PurchaseRealProductAsync.MoveNext (skip CheckPurchasingAlert)',
        'desc': 'Fast purchase — skip CheckPurchasingAlert call, awaiter, and CESA dialog',
        'rva': 0x2831CAC,
        'bytes': struct.pack('<I', 0x14000019),
    },
    {
        'name': 'Initialize.MoveNext (skip _initialized check)',
        'desc': 'Fast purchase — NOP the cbz so Initialize always proceeds (skip ~8s GP timeout)',
        'rva': 0x2830834,
        'bytes': NOP,
    },
]

def prologue(*words: int) -> bytes:
    """Original first instructions of a method, checked before patching it."""
    return struct.pack(f'<{len(words)}I', *words)

# Tracking SDKs. Each method returns void, so an early ret is safe.
TRACKING_IL2CPP_PATCHES = [
    {
        'name': 'Adjust.start',
        'desc': 'Adjust: never start the SDK',
        'rva': 0x3FE22DC,
        'bytes': RET,
        'expected': prologue(0xA9BE4FF4, 0xA9017BFD, 0x910043FD, 0xD001CDB4),
    },
    {
        'name': 'Adjust.trackEvent',
        'desc': 'Adjust: drop events',
        'rva': 0x3FE3810,
        'bytes': RET,
        'expected': prologue(0xA9BE4FF4, 0xA9017BFD, 0x910043FD, 0xB001CDB4),
    },
    {
        'name': 'Adjust.gdprForgetMe',
        'desc': 'Adjust: skip forget-me request',
        'rva': 0x3FE4368,
        'bytes': RET,
        'expected': prologue(0xF81E0FF3, 0xA9017BFD, 0x910043FD, 0x9001CDB3),
    },
    {
        'name': 'Crashlytics.Log',
        'desc': 'Crashlytics: drop breadcrumbs',
        'rva': 0x538D19C,
        'bytes': RET,
        'expected': prologue(0xA9BE4FF4, 0xA9017BFD, 0x910043FD, 0xD00130B4),
    },
    {
        'name': 'Crashlytics.SetUserId',
        'desc': "Crashlytics: don't attach the game user ID",
        'rva': 0x538D214,
        'bytes': RET,
        'expected': prologue(0xA9BE4FF4, 0xA9017BFD, 0x910043FD, 0xD00130B4),
    },
    {
        'name': 'FirebaseAnalytics.SetAnalyticsCollectionEnabled',
        'desc': "Analytics: game can't turn collection back on",
        'rva': 0x53E84A0,
        'bytes': RET,
        'expected': prologue(0xA9BE4FF4, 0xA9017BFD, 0x910043FD, 0x90012DF4),
    },
]

# Meta-data flags for SDKs that start from the manifest, before any game code runs.
TRACKING_META_DATA = [
    # Firebase. "deactivated" is permanent and overrides the game's own SetAnalyticsCollectionEnabled call.
    ('firebase_analytics_collection_deactivated', 'true'),
    ('firebase_analytics_collection_enabled', 'false'),
    ('google_analytics_adid_collection_enabled', 'false'),
    ('firebase_crashlytics_collection_enabled', 'false'),
    ('firebase_messaging_auto_init_enabled', 'false'),
    # Facebook app events.
    ('com.facebook.sdk.AutoInitEnabled', 'false'),
    ('com.facebook.sdk.AutoLogAppEventsEnabled', 'false'),
    ('com.facebook.sdk.AdvertiserIDCollectionEnabled', 'false'),
]

# Startup hooks of SDKs the game never calls (ads) or only needs for Facebook account transfer,
# which can't work on a private server. Firebase providers stay: the game calls Firebase at boot.
TRACKING_COMPONENTS = [
    'com.facebook.internal.FacebookInitProvider',
    'com.facebook.ads.AudienceNetworkContentProvider',
    'com.google.android.gms.ads.MobileAdsInitProvider',
    'com.google.android.gms.ads.AdService',
    'com.applovin.sdk.AppLovinInitProvider',
    'com.applovin.impl.sdk.utils.AppKilledService',
    'com.applovin.impl.adview.activity.FullscreenAdService',
    'com.ironsource.lifecycle.IronsourceLifecycleProvider',
    'com.adjust.sdk.AdjustReferrerReceiver',
]

# Without it, Android 13+ returns an all-zero advertising ID to every SDK.
AD_ID_PERMISSION = 'com.google.android.gms.permission.AD_ID'

# ── Helper functions ──────────────────────────────────────────────────────────
def run(cmd, check=True, shell=False, env=None, cwd=None):
    printable = cmd if isinstance(cmd, str) else ' '.join(str(part) for part in cmd)
    print(f'$ {printable}')
    return subprocess.run(cmd, check=check, text=True, shell=shell, env=env, cwd=cwd)

def ensure_dir(path: Path) -> Path:
    path.mkdir(parents=True, exist_ok=True)
    return path

def movz_w_imm16(rd: int, imm: int) -> bytes:
    """Encode ARM64 MOVZ W<rd>, #imm for a 16-bit immediate."""
    assert 0 <= rd <= 30 and 0 <= imm <= 0xFFFF
    return struct.pack('<I', 0x52800000 | (imm << 5) | rd)


# ── Tool installation ─────────────────────────────────────────────────────────
def install_apktool():
    ensure_dir(TOOLS_DIR)
    jar_path = TOOLS_DIR / f'apktool_{APKTOOL_VERSION}.jar'
    wrapper_path = TOOLS_DIR / 'apktool'

    if not jar_path.exists():
        run([
            'wget', '-q', '-O', str(jar_path),
            f'https://github.com/iBotPeaches/Apktool/releases/download/v{APKTOOL_VERSION}/apktool_{APKTOOL_VERSION}.jar',
        ])

    wrapper_path.write_text(
        '#!/usr/bin/env bash\n'
        f'exec java -jar "{jar_path}" "$@"\n',
        encoding='utf-8',
    )
    wrapper_path.chmod(wrapper_path.stat().st_mode | stat.S_IEXEC)
    os.environ['PATH'] = f'{TOOLS_DIR}:{os.environ["PATH"]}'

def install_android_commandline_tools() -> Path:
    ensure_dir(TOOLS_DIR)
    ensure_dir(ANDROID_SDK_ROOT)

    zip_path = TOOLS_DIR / 'commandlinetools.zip'
    temp_dir = TOOLS_DIR / 'cmdline-tools-temp'
    cmdline_root = ANDROID_SDK_ROOT / 'cmdline-tools'
    latest_dir = cmdline_root / 'latest'

    if not latest_dir.exists():
        if temp_dir.exists():
            shutil.rmtree(temp_dir)
        run(['wget', '-q', '-O', str(zip_path), ANDROID_CMDLINE_TOOLS_URL])
        ensure_dir(temp_dir)
        run(['unzip', '-q', str(zip_path), '-d', str(temp_dir)])
        ensure_dir(cmdline_root)
        extracted_dir = temp_dir / 'cmdline-tools'
        if latest_dir.exists():
            shutil.rmtree(latest_dir)
        shutil.move(str(extracted_dir), str(latest_dir))

    return latest_dir

def install_android_build_tools():
    latest_dir = install_android_commandline_tools()
    sdkmanager = latest_dir / 'bin' / 'sdkmanager'

    env = os.environ.copy()
    env['ANDROID_SDK_ROOT'] = str(ANDROID_SDK_ROOT)
    env['JAVA_HOME'] = '/usr/lib/jvm/java-17-openjdk-amd64'
    env['PATH'] = f'{latest_dir / "bin"}:{env["PATH"]}'

    run(f'yes | "{sdkmanager}" --sdk_root="{ANDROID_SDK_ROOT}" --licenses >/dev/null', check=False, shell=True, env=env)
    run([
        str(sdkmanager),
        f'--sdk_root={ANDROID_SDK_ROOT}',
        f'build-tools;{ANDROID_BUILD_TOOLS}',
        'platform-tools',
    ], env=env)

    build_tools_dir = ANDROID_SDK_ROOT / 'build-tools' / ANDROID_BUILD_TOOLS
    os.environ['ANDROID_SDK_ROOT'] = str(ANDROID_SDK_ROOT)
    os.environ['PATH'] = f'{build_tools_dir}:{latest_dir / "bin"}:{os.environ["PATH"]}'

def install_dependencies():
    ensure_dir(WORK_ROOT)
    ensure_dir(TOOLS_DIR)

    run(['apt-get', 'update'])
    run([
        'apt-get', 'install', '-y',
        'openjdk-17-jdk-headless',
        'wget',
        'unzip',
        'ca-certificates',
    ])

    install_apktool()
    install_android_build_tools()

    missing = [tool for tool in ('apktool', 'apksigner', 'zipalign', 'keytool') if shutil.which(tool) is None]
    if missing:
        raise RuntimeError(f'Missing required tools after installation: {missing}')

    print('\nInstalled tool paths:')
    for tool in ('apktool', 'apksigner', 'zipalign', 'keytool'):
        print(f'  {tool}: {shutil.which(tool)}')

    print('\nRequested versions:')
    print(f'  apktool: {APKTOOL_VERSION}')
    print(f'  Android build-tools: {ANDROID_BUILD_TOOLS}')

# ── Patch helpers ─────────────────────────────────────────────────────────────
@dataclass
class RuntimeConfig:
    protocol: str
    host: str
    http_port: str = ''
    grpc_port: str = ''

    @property
    def web_base_url(self) -> str:
        base = f'{self.protocol}://{self.host}'
        if self.http_port:
            base += f':{self.http_port}'
        return base

    @property
    def grpc_port_int(self) -> int:
        return int(self.grpc_port) if self.grpc_port else 443

def require_file(path: Path):
    if not path.is_file():
        raise FileNotFoundError(f'Not found: {path}')

def build_replacements(cfg: RuntimeConfig):
    replacements = [
        (f'api.app.{GAME_NAME}.com', cfg.host),
        (
            f'https://web.app.{GAME_NAME}.com/assets/release/{{0}}/database.bin',
            f'{cfg.web_base_url}/assets/release/{{0}}/database.bin',
        ),
        (f'https://web.app.{GAME_NAME}.com', cfg.web_base_url),
        (f'https://resources-api.app.{GAME_NAME}.com/', f'{cfg.web_base_url}/'),
    ]

    for old_value, new_value in replacements:
        if len(new_value.encode('utf-8')) > len(old_value.encode('utf-8')):
            raise ValueError(
                'Replacement too long: '
                f'{old_value!r} -> {new_value!r}. '
                'Use a shorter host name or omit the port.'
            )

    return replacements

def patch_metadata_strings(meta_path: Path, replacements) -> int:
    data = bytearray(meta_path.read_bytes())

    magic = struct.unpack_from('<I', data, 0)[0]
    if magic != METADATA_MAGIC:
        raise RuntimeError(f'Bad metadata magic: 0x{magic:08X}')

    version = struct.unpack_from('<i', data, 4)[0]
    print(f'  metadata v{version}, {len(data)} bytes')

    sl_off, sl_size = struct.unpack_from('<II', data, HDR_STRING_LITERAL_OFF)
    sld_off, sld_size = struct.unpack_from('<II', data, HDR_STRING_LITERAL_DATA_OFF)
    n_entries = sl_size // 8
    print(f'  stringLiteral: {n_entries} entries @ 0x{sl_off:X}')
    print(f'  stringLiteralData: {sld_size} bytes @ 0x{sld_off:X}')

    patched = 0
    for old_str, new_str in replacements:
        old_bytes = old_str.encode('utf-8')
        new_bytes = new_str.encode('utf-8')

        blob_pos = data.find(old_bytes, sld_off, sld_off + sld_size)
        if blob_pos < 0:
            print(f'  [!] NOT FOUND in blob: {old_str!r}')
            continue

        data_index = blob_pos - sld_off
        entry_found = False
        for i in range(n_entries):
            entry_offset = sl_off + i * 8
            entry_len, entry_index = struct.unpack_from('<II', data, entry_offset)
            if entry_index == data_index and entry_len == len(old_bytes):
                struct.pack_into('<I', data, entry_offset, len(new_bytes))
                entry_found = True
                print(f'  entry #{i}: length {entry_len} -> {len(new_bytes)}')
                break

        if not entry_found:
            print(f'  [!] No table entry found for {old_str!r} (dataIndex=0x{data_index:X})')
            continue

        data[blob_pos:blob_pos + len(old_bytes)] = new_bytes + (b'\x00' * (len(old_bytes) - len(new_bytes)))
        print(f'  PATCHED: {old_str!r} -> {new_str!r}')
        patched += 1

    meta_path.write_bytes(data)
    return patched

def patch_manifest(manifest_path: Path) -> bool:
    text = manifest_path.read_text(encoding='utf-8')
    changed = False

    if 'networkSecurityConfig' not in text:
        new_attr = 'android:networkSecurityConfig="@xml/network_security_config"'
        text, count = re.subn(r'<application\b', f'<application {new_attr}', text, count=1)
        if count != 1:
            raise RuntimeError('Could not find <application> tag in AndroidManifest.xml')
        print(f'  added {new_attr}')
        changed = True
    else:
        print('  networkSecurityConfig already present')

    if changed:
        manifest_path.write_text(text, encoding='utf-8')

    return True

def fix_extract_native_libs(manifest_path: Path) -> bool:
    # apktool rebuilds can leave native libs in a state the device can't load directly,
    # so make Android extract them on install instead.
    text = manifest_path.read_text(encoding='utf-8')
    if 'android:extractNativeLibs="false"' not in text:
        print('  extractNativeLibs not set to false; nothing to change')
        return False
    text = text.replace('android:extractNativeLibs="false"', 'android:extractNativeLibs="true"')
    manifest_path.write_text(text, encoding='utf-8')
    print('  extractNativeLibs: false -> true')
    return True

def _element_pattern(tags: str, name: str) -> str:
    """Matches one <tag android:name="name" .../> or <tag ...>...</tag> line block, with its indent."""
    return (
        rf'[ \t]*<({tags})\b(?=[^>]*\bandroid:name="{re.escape(name)}")[^>]*?'
        r'(?:/>|>.*?</\1>)[ \t]*\r?\n?'
    )

def disable_tracking_in_manifest(manifest_path: Path):
    text = manifest_path.read_text(encoding='utf-8')

    for name, value in TRACKING_META_DATA:
        pattern = rf'<meta-data\b(?=[^>]*\bandroid:name="{re.escape(name)}")[^>]*>'
        match = re.search(pattern, text)
        if match:
            tag = re.sub(r'android:value="[^"]*"', f'android:value="{value}"', match.group(0))
            text = text[:match.start()] + tag + text[match.end():]
            print(f'  meta-data {name} = {value}')
        else:
            new_tag = f'<meta-data android:name="{name}" android:value="{value}"/>'
            text, count = re.subn(r'([ \t]*)</application>', rf'\1    {new_tag}\n\1</application>', text, count=1)
            if count != 1:
                raise RuntimeError('Could not find </application> in AndroidManifest.xml')
            print(f'  meta-data {name} = {value} (added)')

    for name in TRACKING_COMPONENTS:
        text, count = re.subn(_element_pattern('provider|service|receiver', name), '', text, flags=re.DOTALL)
        print(f'  removed {name}' if count else f'  {name} not present')

    text, count = re.subn(_element_pattern('uses-permission', AD_ID_PERMISSION), '', text, flags=re.DOTALL)
    print(f'  removed permission {AD_ID_PERMISSION}' if count else f'  {AD_ID_PERMISSION} not present')

    manifest_path.write_text(text, encoding='utf-8')

NETWORK_SECURITY_CONFIG = '''<?xml version="1.0" encoding="utf-8"?>
<network-security-config>
    <base-config cleartextTrafficPermitted="true" />
</network-security-config>
'''

def create_network_security_config(res_xml_dir: Path) -> bool:
    res_xml_dir.mkdir(parents=True, exist_ok=True)
    output_path = res_xml_dir / 'network_security_config.xml'
    output_path.write_text(NETWORK_SECURITY_CONFIG, encoding='utf-8')
    print(f'  wrote {output_path}')
    return True

def patch_libil2cpp(so_path: Path, patches: list) -> int:
    patched = 0
    with so_path.open('r+b') as handle:
        file_size = handle.seek(0, 2)
        for patch in patches:
            rva = patch['rva']
            patch_bytes = patch['bytes']
            if rva + len(patch_bytes) > file_size:
                print(f"  [!] SKIP {patch['name']}: RVA 0x{rva:X} beyond file size")
                continue

            expected = patch.get('expected')
            if expected is not None:
                handle.seek(rva)
                actual = handle.read(len(expected))
                if actual != expected:
                    print(f"  [!] SKIP {patch['name']}: unexpected bytes at 0x{rva:X} ({actual.hex()}), wrong game version?")
                    continue

            handle.seek(rva)
            original = handle.read(len(patch_bytes))
            handle.seek(rva)
            handle.write(patch_bytes)

            patched += 1
            print(f"  {patch['name']} @ 0x{rva:X}: {original.hex()} -> {patch_bytes.hex()}")
            print(f"    {patch['desc']}")

    return patched

def ensure_debug_keystore() -> Path:
    keystore_path = TOOLS_DIR / 'debug.keystore'
    if keystore_path.exists():
        return keystore_path

    run([
        'keytool',
        '-genkeypair',
        '-v',
        '-keystore', str(keystore_path),
        '-storepass', 'android',
        '-alias', 'androiddebugkey',
        '-keypass', 'android',
        '-keyalg', 'RSA',
        '-keysize', '2048',
        '-validity', '10000',
        '-dname', 'CN=Android Debug,O=Android,C=US',
        '-noprompt',
    ])
    return keystore_path

def patch_uploaded_apk(uploaded_apk: Path, cfg: RuntimeConfig) -> Path:
    job_dir = WORK_ROOT / 'job'
    decoded_dir = job_dir / 'decoded'
    unsigned_apk = job_dir / f'{uploaded_apk.stem}-unsigned.apk'
    aligned_apk = job_dir / f'{uploaded_apk.stem}-aligned.apk'
    signed_apk = job_dir / f'{uploaded_apk.stem}-patched.apk'

    if job_dir.exists():
        shutil.rmtree(job_dir)
    job_dir.mkdir(parents=True, exist_ok=True)

    print(f'[*] Decompiling {uploaded_apk.name} ...')
    run(['apktool', 'd', '-f', str(uploaded_apk), '-o', str(decoded_dir)])

    metadata_path = decoded_dir / 'assets/bin/Data/Managed/Metadata/global-metadata.dat'
    so_path = decoded_dir / 'lib/arm64-v8a/libil2cpp.so'
    manifest_path = decoded_dir / 'AndroidManifest.xml'
    res_xml_dir = decoded_dir / 'res/xml'

    for required_path in (metadata_path, so_path, manifest_path):
        require_file(required_path)

    replacements = build_replacements(cfg)
    print(f'\n[*] Patching for server {cfg.host} (web={cfg.web_base_url}, gRPC={cfg.host}:{cfg.grpc_port_int})')

    print('\n[1] Patching global-metadata.dat string literals ...')
    patched_strings = patch_metadata_strings(metadata_path, replacements)
    print(f'    {patched_strings}/{len(replacements)} strings patched')

    print('\n[2] Patching libil2cpp.so (SSL bypass, encryption, Octo, purchases) ...')
    patched_methods = patch_libil2cpp(so_path, IL2CPP_PATCHES)
    print(f'    {patched_methods}/{len(IL2CPP_PATCHES)} methods patched')

    print('\n[3] Fixing extractNativeLibs in AndroidManifest.xml ...')
    fix_extract_native_libs(manifest_path)

    if cfg.protocol == 'http':
        print('\n[4] Enabling cleartext traffic in AndroidManifest.xml ...')
        patch_manifest(manifest_path)

        print('\n[5] Writing network_security_config.xml ...')
        create_network_security_config(res_xml_dir)
    else:
        print('\n[4] HTTPS selected; skipping cleartext network security changes')

    if disable_tracking:
        print('\n[6] Disabling tracking and analytics ...')
        patched_tracking = patch_libil2cpp(so_path, TRACKING_IL2CPP_PATCHES)
        print(f'    {patched_tracking}/{len(TRACKING_IL2CPP_PATCHES)} tracking methods patched')
        disable_tracking_in_manifest(manifest_path)
    else:
        print('\n[6] disable_tracking is off; leaving tracking SDKs alone')

    print('\n[7] Rebuilding APK ...')
    run(['apktool', 'b', str(decoded_dir), '-o', str(unsigned_apk)])

    print('\n[8] Aligning APK ...')
    # -p page-aligns native libs so they also load if left uncompressed.
    run(['zipalign', '-p', '-f', '4', str(unsigned_apk), str(aligned_apk)])

    print('\n[9] Signing APK ...')
    keystore_path = ensure_debug_keystore()
    if signed_apk.exists():
        signed_apk.unlink()
    shutil.copy2(aligned_apk, signed_apk)
    run([
        'apksigner', 'sign',
        '--ks', str(keystore_path),
        '--ks-pass', 'pass:android',
        '--key-pass', 'pass:android',
        '--ks-key-alias', 'androiddebugkey',
        str(signed_apk),
    ])

    print('\n[10] Verifying signature ...')
    run(['apksigner', 'verify', '--verbose', str(signed_apk)])

    print(f'\n[+] Finished: {signed_apk}')
    return signed_apk

# ── Validate config ───────────────────────────────────────────────────────────
server_host = server_host.strip()
http_port = http_port.strip()
grpc_port = grpc_port.strip()

if not server_host:
    raise ValueError('server_host cannot be empty')

if http_port and not http_port.isdigit():
    raise ValueError('http_port must be blank or numeric')

if grpc_port and not grpc_port.isdigit():
    raise ValueError('grpc_port must be blank or numeric')

grpc_port_val = int(grpc_port) if grpc_port else 443
if not (1 <= grpc_port_val <= 65535):
    raise ValueError(f'grpc_port must be between 1 and 65535, got {grpc_port_val}')

cfg = RuntimeConfig(protocol=protocol, host=server_host, http_port=http_port, grpc_port=grpc_port)
print(f'Configured web base URL: {cfg.web_base_url}')
print(f'Configured gRPC: {cfg.host}:{cfg.grpc_port_int}')
IL2CPP_PATCHES = BASE_IL2CPP_PATCHES + [
    {
        'name': 'NetworkConfig.get_ServerPort',
        'desc': f'gRPC port override — MOVZ W0, #{cfg.grpc_port_int} (return configured port)',
        'rva': 0x361D548,
        'bytes': movz_w_imm16(0, cfg.grpc_port_int) + RET,
    },
    {
        'name': 'InitializeApiClient.OnStateBegin (port override)',
        'desc': f'conductor port override — MOVZ W2, #{cfg.grpc_port_int}',
        'rva': 0x2DEAF68,
        'bytes': movz_w_imm16(2, cfg.grpc_port_int),
    },
    {
        'name': 'CalculatorNetworking.InitializeApiClient (port override)',
        'desc': f'catch-all port override — MOVZ W20, #{cfg.grpc_port_int}',
        'rva': 0x2E1B278,
        'bytes': movz_w_imm16(20, cfg.grpc_port_int),
    },
]


# ── Install dependencies ──────────────────────────────────────────────────────
install_dependencies()

# ── Acquire APK ──────────────────────────────────────────────────────────────
ensure_dir(WORK_ROOT)

if apk_source == 'upload':
    print('\nPlease upload your APK file...')
    try:
        from google.colab import files as colab_files
        uploaded = colab_files.upload()
        if not uploaded:
            raise RuntimeError('No file uploaded')
        apk_filename = next(iter(uploaded))
        input_apk = WORK_ROOT / Path(apk_filename).name
        input_apk.write_bytes(uploaded[apk_filename])
    except ImportError:
        raise RuntimeError('File upload is only available in Google Colab')
    print(f'Uploaded {input_apk.name} ({input_apk.stat().st_size:,} bytes)')
else:
    apk_url = apk_url.strip()
    if not apk_url:
        raise ValueError('apk_url cannot be empty when apk_source is "url"')
    url_filename = Path(urllib.parse.unquote(apk_url.split('/')[-1]))
    input_name = url_filename.name if url_filename.suffix else 'input.apk'
    input_apk = WORK_ROOT / input_name
    print(f'\nDownloading APK from {apk_url} ...')
    run(['wget', '-q', '--show-progress', '-O', str(input_apk), apk_url])
    if not input_apk.exists():
        raise RuntimeError('Download failed')
    print(f'Downloaded {input_apk.name} ({input_apk.stat().st_size:,} bytes)')

# ── Patch and export ──────────────────────────────────────────────────────────
output_apk = patch_uploaded_apk(input_apk, cfg)
print(f'Patched APK: {output_apk}')

try:
    from google.colab import files
    files.download(str(output_apk))
except Exception:
    print('Browser download failed; patched APK is at:', output_apk)